# 00 — Preflight and lab region

**Requirement.** Before any hosted-agent test, establish (1) that the lab subscription, identity, policies,
providers, model and quota allow the scenario in the chosen lab region, and (2) what Microsoft Learn states
about hosted-agent regional availability, including North Europe (checklist criterion C05).

**What this notebook does.** Read-only calls to the subscription in `.env` and public Learn pages.
It provisions nothing, cleans up nothing, and changes no Azure configuration. Every check comes from the
`hapq` package; this notebook only calls it and displays results.

**How to read results.** `observed` = live call by this kit, `documented` = Learn statement on the fetch date.
Neither is a demonstration of hosted-agent capacity: that requires the Phase A lab (pending approval).

## 1. Configuration

In [ ]:
from datetime import datetime, timezone
from IPython.display import Markdown, display

from hapq import api, report
from hapq.config import load_config
from hapq.redact import redact_text

cfg = load_config()  # subscription id from .env; only a fingerprint is ever shown
RECORD = True        # write results to results/<run_id>/ (local, git-ignored)

display(Markdown(f'''
| Setting | Value |
|---|---|
| Subscription | `{cfg.subscription_fingerprint}` (fingerprint, the id is never displayed) |
| Resource group | `{cfg.resource_group}` |
| Lab region | `{cfg.lab_region}` |
| Region tracked separately | `{cfg.required_region}` |
| Model | `{cfg.model.name}` {cfg.model.version}, {cfg.model.sku}, {cfg.model.capacity}K TPM |
| Declared execution point | `{cfg.execution_point}` |
| Config hash | `{cfg.config_hash}` |
| Executed at (UTC) | {datetime.now(timezone.utc).isoformat(timespec="seconds")} |
'''))

## 2. Execution — read-only preflight

Expected: identity authenticated on the `.env` subscription, kit resource group present and tagged, providers
registered, no policy excluding the lab region, provisioning + Foundry data-plane roles, model offered with
enough quota. `PF-ARM-LOCATIONS-REQUIRED` is expected to stay `NOT_TESTED`: ARM metadata cannot prove
hosted-agent support in North Europe.

In [ ]:
preflight_results, preflight_run = api.run_preflight(cfg, record=RECORD)
display(Markdown(api.as_markdown(preflight_results)))
print("run:", preflight_run.name if preflight_run else "not recorded")

## 3. Execution — documentary region checks (C05)

Fetches the current Learn pages and evaluates the region assertions from `config/doc_assertions.toml`.

In [ ]:
doc_results, doc_run = api.run_docs(cfg, criteria={"C05.a", "C05.b", "C05.c", "C05.d"}, record=RECORD)
display(Markdown(api.as_markdown(doc_results, columns=("check_id", "status", "evidence", "observed"))))
print("run:", doc_run.name if doc_run else "not recorded")

## 4. Expected vs observed

In [ ]:
expected = {
    "PF-AUTH": "PASS", "PF-RG": "PASS", "PF-POLICIES": "PASS", "PF-ROLES": "PASS",
    "PF-MODEL-AVAILABLE": "PASS", "PF-MODEL-QUOTA": "PASS",
    "PF-ARM-LOCATIONS-REQUIRED": "NOT_TESTED",
    "DOC-C05-WESTEUROPE": "PASS",
    "DOC-C05-NORTHEUROPE": "FAIL",   # initial checklist claim: not listed
}
observed = {r.check_id: r.status.value for r in preflight_results + doc_results}
rows = "\n".join(f"| {k} | {v} | {observed.get(k, 'missing')} | {'yes' if observed.get(k) == v else '**NO**'} |"
                  for k, v in expected.items())
display(Markdown("| Check | Expected | Observed | Match |\n|---|---|---|---|\n" + rows))

## 5. Criteria matrix excerpt and freshness

The matrix keeps the documentary layer and the execution layer apart; verdicts use execution results only,
per environment. Results older than the configured window, or produced with another configuration or kit
version, are flagged STALE.

In [ ]:
table, _ = report.build_matrix(cfg)
lines = ["| Criterion | Layer | Check | Env@point | Status | Run | Freshness |", "|---|---|---|---|---|---|---|"]
for c in table:
    if c["id"].startswith("C05"):
        for layer, rows in (("documentary", c["docs"]), ("execution", c["preflight"] + c["cells"])):
            for r in rows:
                fresh = "-" if not r.timestamp else ("STALE: " + ", ".join(r.stale) if r.stale else "current")
                lines.append(f"| {c['id']} | {layer} | {r.check_id} | {r.run_label} | {r.status} | {r.run_id or '-'} | {fresh} |")
display(Markdown(redact_text("\n".join(lines))))
display(Markdown("**Verdicts (execution only):** " + "; ".join(
    f"{c['id']}: A = {c['verdict_A']}, B = {c['verdict_B']}" for c in table if c["id"].startswith("C05"))))

## 6. Limits

- Documented availability is not capacity: West Europe becomes *demonstrated* only when a hosted agent version
  reaches `active` there (scenario `S01-GA-CREATE`, after approval).
- A success in West Europe does not validate North Europe. North Europe is not deployed to by design; its status
  rests on documentation and is re-evaluated each time this notebook runs.
- Quota is not guaranteed regional capacity; catalog listing is not deployment success.
- The policy check evaluates location restrictions only; network-related deny policies surface at deployment.
- Learn pages change: each documentary result stores the page hash and fetch time.